# 5차시. 데이터 변환과 파생 변수 생성

## 1. 5차시 학습목표
열 계산, map·apply, 조건별 값, 구간 나누기로 파생 변수를 만드는 이번 차시 목표를 확인합니다.

## 2. 선수지식 — 함수 정의와 타입 변환
달러를 원으로 바꾸는 함수를 만들어 보고, 조건 결과 True/False를 1/0으로 바꿉니다.

In [1]:
import pandas as pd                          # 판다스 불러오기
import numpy as np                           # np.where · np.select 사용

tips = pd.read_csv("data/raw/tips.csv")      # tips 데이터 불러오기

def to_won(dollar):                          # 함수 정의: 달러 → 원
    return dollar * 1400

print(to_won(10))

is_big = tips["tip"] >= 5                    # 조건 → True/False
is_big_num = is_big.astype(int)              # True/False → 1/0
print(is_big.head(3).tolist())
print(is_big_num.head(3).tolist())

14000
[False, False, False]
[0, 0, 0]


## 3. 파생 변수 — 열끼리 계산
팁 비율, 1인당 금액, 팁 포함 결제액 열을 계산해 추가합니다.

In [7]:
# 팁 비율
tips["tip_rate"] = tips["tip"] / tips["total_bill"] * 100

# 1인당 결제금액
tips["per_person"] = tips["total_bill"] / tips["size"]

# 팁을 포함한 결제금액
tips["total_pay"] = tips["total_bill"] + tips["tip"]

tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20
...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57


## 4. map — 딕셔너리로 값 바꾸기
성별·요일을 한글로 바꾸고, 사전에 없는 값이 NaN이 되는 것을 확인합니다.

In [12]:
sex_map = {"Female": "여성", "Male": "남성"}             # 기존 값 → 새 값
tips["sex_kr"] = tips["sex"].map(sex_map)               # 딕셔너리로 값 바꾸기

day_map = {"Thur": "목", "Fri": "금", "Sat": "토"}       # 일부러 "Sun"을 빠뜨림
tips["day_kr"] = tips["day"].map(day_map)

print(tips[["sex", "sex_kr", "day", "day_kr"]].head(3))
print("day_kr 결측:", tips["day_kr"].isna().sum())

      sex sex_kr  day day_kr
0  Female     여성  Sun    NaN
1    Male     남성  Sun    NaN
2    Male     남성  Sun    NaN
day_kr 결측: 76


## 5. map — 함수로 값 바꾸기
인원 수를 혼밥·2인·단체로 나누는 함수를 만들어 map으로 적용합니다.

In [16]:
def f01(x):
    if x == 1:
        return "혼밥"
    elif x == 2:
        return "2인"
    else:
        return "단체"

tips["table_type"] = tips["size"].map( f01 )

tips["table_type"].value_counts()

table_type
2인    156
단체     84
혼밥      4
Name: count, dtype: int64

## 6. apply — 추가 인자가 있는 함수 적용
부가세 함수를 apply로 적용하고, 세율을 인자로 바꿔 넘겨 봅니다.

In [23]:
def f01(tb , rate=0.1):
    return tb + tb*rate

tips["bill_vat10"] = tips["total_bill"].apply( f01 )
tips["bill_vat5"] = tips["total_bill"].apply( f01 , rate=0.05 )

tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,sex_kr,day_kr,table_type,bill_vat10,bill_vat5
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,여성,NaN,2인,18.689,17.8395
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,남성,NaN,단체,11.374,10.8570
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,남성,NaN,단체,23.111,22.0605
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,남성,NaN,2인,26.048,24.8640
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,여성,NaN,단체,27.049,25.8195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,남성,토,단체,31.933,30.4815
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,여성,토,2인,29.898,28.5390
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,남성,토,2인,24.937,23.8035
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,남성,토,2인,19.602,18.7110


## 7. apply(axis=1) — 여러 열을 보고 판단
결제 금액과 인원 수를 함께 보고 고객 등급을 매깁니다.

In [40]:
def f01(row):
    tb = row["total_bill"]
    s = row["size"]
    if tb > 25 and s >= 4:
        return "VIP team"
    elif tb > 25:
        return "VIP"
    else:
        return "Normal"

tips["grade"] = tips.apply( f01 , axis=1)
tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,sex_kr,day_kr,table_type,bill_vat10,bill_vat5,grade
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,여성,NaN,2인,18.689,17.8395,Normal
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,남성,NaN,단체,11.374,10.8570,Normal
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,남성,NaN,단체,23.111,22.0605,Normal
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,남성,NaN,2인,26.048,24.8640,Normal
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,여성,NaN,단체,27.049,25.8195,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,남성,토,단체,31.933,30.4815,VIP
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,여성,토,2인,29.898,28.5390,VIP
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,남성,토,2인,24.937,23.8035,Normal
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,남성,토,2인,19.602,18.7110,Normal


## 8. 조건 2갈래 — np.where
팁 5 이상 여부와 주말 여부를 np.where로 만듭니다.

In [42]:
tips["is_good"] = np.where( tips["tip"] >= 5 , "Y" , "N" )
tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,sex_kr,day_kr,table_type,bill_vat10,bill_vat5,grade,is_good
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,여성,NaN,2인,18.689,17.8395,Normal,N
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,남성,NaN,단체,11.374,10.8570,Normal,N
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,남성,NaN,단체,23.111,22.0605,Normal,N
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,남성,NaN,2인,26.048,24.8640,Normal,N
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,여성,NaN,단체,27.049,25.8195,Normal,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,남성,토,단체,31.933,30.4815,VIP,Y
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,여성,토,2인,29.898,28.5390,VIP,N
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,남성,토,2인,24.937,23.8035,Normal,N
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,남성,토,2인,19.602,18.7110,Normal,N


In [44]:
x = tips["day"].isin( [ "Sat" , "Sun"] )
tips["is_weekend"] = np.where( x , "happy", "sad")
tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,sex_kr,day_kr,table_type,bill_vat10,bill_vat5,grade,is_good,is_weekend
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,여성,NaN,2인,18.689,17.8395,Normal,N,happy
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,남성,NaN,단체,11.374,10.8570,Normal,N,happy
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,남성,NaN,단체,23.111,22.0605,Normal,N,happy
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,남성,NaN,2인,26.048,24.8640,Normal,N,happy
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,여성,NaN,단체,27.049,25.8195,Normal,N,happy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,남성,토,단체,31.933,30.4815,VIP,Y,happy
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,여성,토,2인,29.898,28.5390,VIP,N,happy
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,남성,토,2인,24.937,23.8035,Normal,N,happy
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,남성,토,2인,19.602,18.7110,Normal,N,happy


## 9. 조건 여러 갈래 — numpy select
팁 비율을 기준으로 A·B·C·D 네 등급을 매깁니다.

In [53]:
cond = [
    tips["tip_rate"] >= 20 ,
    tips["tip_rate"] >= 15 ,
    tips["tip_rate"] >= 10 ,
]
choice = ["A" , "B" , "C"]
tips["tip_grade"] = np.select(cond , choice , default="D")
x = tips["tip_grade"].value_counts()
x.sort_index()

tip_grade
A    39
B    97
C    81
D    27
Name: count, dtype: int64

## 10. 구간 나누기 — pd.cut
결제 금액을 4개 구간으로 나누고 구간 이름을 붙입니다.

In [60]:
tips["bill_band"] = pd.cut(
    tips["total_bill"] ,
    bins=[0,10,20,30,60]
)

tips["bill_band_kr"] = pd.cut(
    tips["total_bill"] ,
    bins=[0,10,20,30,60] ,
    labels=["구간01" , "구간02" , "구간03" , "구간04"]
)

tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,...,day_kr,table_type,bill_vat10,bill_vat5,grade,is_good,is_weekend,tip_grade,bill_band,bill_band_kr
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,...,NaN,2인,18.689,17.8395,Normal,N,happy,D,"(10, 20]",구간02
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,...,NaN,단체,11.374,10.8570,Normal,N,happy,B,"(10, 20]",구간02
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,...,NaN,단체,23.111,22.0605,Normal,N,happy,B,"(20, 30]",구간03
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,...,NaN,2인,26.048,24.8640,Normal,N,happy,C,"(20, 30]",구간03
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,...,NaN,단체,27.049,25.8195,Normal,N,happy,C,"(20, 30]",구간03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,...,토,단체,31.933,30.4815,VIP,Y,happy,A,"(20, 30]",구간03
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,...,토,2인,29.898,28.5390,VIP,N,happy,D,"(20, 30]",구간03
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,...,토,2인,24.937,23.8035,Normal,N,happy,D,"(20, 30]",구간03
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,...,토,2인,19.602,18.7110,Normal,N,happy,D,"(10, 20]",구간02


## 11. cut 경계 주의 — right · include_lowest
경계값과 범위 밖 값이 어떻게 처리되는지 include_lowest 옵션과 비교합니다.

In [66]:
s = pd.Series([0, 10, 10.01, 60, 70])

cut_basic = pd.cut(s, bins=[0, 10, 60])                          # 0과 70은 범위 밖
cut_basic
print(cut_basic.astype(str).tolist())

cut_lowest = pd.cut(s, bins=[0, 10, 60], include_lowest=True)    # 0 포함
print(cut_lowest.astype(str).tolist())

0             NaN
1     (0.0, 10.0]
2    (10.0, 60.0]
3    (10.0, 60.0]
4             NaN
dtype: category
Categories (2, interval[int64, right]): [(0, 10] < (10, 60]]

## 12. 개수가 같도록 나누기 — pd.qcut
결제 금액을 인원이 같은 4그룹으로 나누고 자동 계산된 경계를 확인합니다.

In [76]:
tips["bill_q"] = pd.qcut(
    tips["total_bill"] ,
    q=4,
    labels=["하" , "중하" , "중상" , "상"]
)

tips

,total_bill,tip,sex,smoker,day,time,size,tip_rate,per_person,total_pay,...,table_type,bill_vat10,bill_vat5,grade,is_good,is_weekend,tip_grade,bill_band,bill_band_kr,bill_q
0,16.99,1.01,Female,No,Sun,Dinner,2,5.944673,8.495000,18.00,...,2인,18.689,17.8395,Normal,N,happy,D,"(10, 20]",구간02,중하
1,10.34,1.66,Male,No,Sun,Dinner,3,16.054159,3.446667,12.00,...,단체,11.374,10.8570,Normal,N,happy,B,"(10, 20]",구간02,하
2,21.01,3.50,Male,No,Sun,Dinner,3,16.658734,7.003333,24.51,...,단체,23.111,22.0605,Normal,N,happy,B,"(20, 30]",구간03,중상
3,23.68,3.31,Male,No,Sun,Dinner,2,13.978041,11.840000,26.99,...,2인,26.048,24.8640,Normal,N,happy,C,"(20, 30]",구간03,중상
4,24.59,3.61,Female,No,Sun,Dinner,4,14.680765,6.147500,28.20,...,단체,27.049,25.8195,Normal,N,happy,C,"(20, 30]",구간03,상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.392697,9.676667,34.95,...,단체,31.933,30.4815,VIP,Y,happy,A,"(20, 30]",구간03,상
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.358352,13.590000,29.18,...,2인,29.898,28.5390,VIP,N,happy,D,"(20, 30]",구간03,상
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.822232,11.335000,24.67,...,2인,24.937,23.8035,Normal,N,happy,D,"(20, 30]",구간03,중상
242,17.82,1.75,Male,No,Sat,Dinner,2,9.820426,8.910000,19.57,...,2인,19.602,18.7110,Normal,N,happy,D,"(10, 20]",구간02,중상


## 13. 종합 실습 — add_features 함수
3차시 정제 데이터에 파생 변수 5개를 추가하는 함수를 만들고 결과를 확인합니다.

In [80]:
titanic = pd.read_csv("data/processed/titanic_clean.csv")

def add_features(df):

    sex_map = {
        "male" : "남성" ,
        "female" : "여성" ,
    }
    df["sex_kr"] = df["sex"].map( sex_map )
    df["is_child"] = np.where( df["age"] <= 13 , 1 , 0)
    df["age_group"] = pd.cut(
        df["age"] ,
        bins=[0,13,19,39,59,100],
        labels=["어린이" , "청소년" , "청년" , "중년" , "노년"]
    )     
    df["fare_level"] = pd.qcut(
        df["fare"] ,
        q=3 ,
        labels=["저가" , "중가" , "고가"]
    )
    class_map = {
        1 : "1등석" ,
        2 : "2등석" ,
        3 : "3등석" ,
    }
    df["class_kr"] = df["pclass"].map( class_map )

    return df

titanic_feat = add_features(titanic)
titanic_feat

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,sex_kr,is_child,age_group,fare_level,class_kr
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,남성,0,청년,저가,3등석
1,1,1,female,38.0,1,0,65.6344,C,First,woman,False,Cherbourg,yes,False,여성,0,청년,고가,1등석
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,여성,0,청년,저가,3등석
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,여성,0,청년,고가,1등석
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,남성,0,청년,저가,3등석
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,Southampton,no,True,남성,0,청년,중가,2등석
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True,여성,0,청소년,고가,1등석
888,0,3,female,28.0,1,2,23.4500,S,Third,woman,False,Southampton,no,False,여성,0,청년,중가,3등석
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True,남성,0,청년,고가,1등석


## 14. 파생 변수로 질문에 답하기 — 연령대별 생존률
어린이와 노년의 생존률을 비교하고, 파생 변수가 포함된 결과를 파일로 저장합니다.

In [105]:
# 어린이 생존율
mask = titanic_feat["is_child"] == 1
child_rate = titanic_feat.loc[mask , "survived"].mean()
child_rate

np.float64(0.5915492957746479)

In [103]:
# 노년 생존율
mask = titanic_feat["age_group"] == "노년"
old_rate = titanic_feat.loc[mask , "survived"].mean()
old_rate

np.float64(0.2692307692307692)

In [107]:
# 파생칼럼 추가된 df 저장하기
titanic_feat.to_csv("data/processed/titanic_features.csv" , index=False)

## 15. 5차시 정리
열 계산, map·apply, 조건별 파생 변수, 구간 나누기를 정리합니다.